# Notebook C2: Chạy QAOA trên IBM Quantum

Đi kèm [docs/C-hardware/ibm-quantum.md](../../docs/C-hardware/ibm-quantum.md) và [Phần A, Mục 6](../../docs/A-translation/06-gate-based.md).

**Notebook chạy được dù bạn có hay không có tài khoản IBM Quantum.** Nếu không kết nối được, nó dùng `FakeKingston`: một mô hình noise mô phỏng chip Heron r2 156 qubit của IBM, chạy hoàn toàn trên máy bạn. Giao diện (`generate_preset_pass_manager`, `SamplerV2`) giống hệt khi chạy máy thật, nên ta thấy được tác động của noise và quá trình transpile.

Chiến lược quan trọng để tiết kiệm thời gian QPU (gói miễn phí giới hạn rất ít): **tối ưu tham số trên trình mô phỏng lý tưởng trước, chỉ chạy mạch cuối cùng trên máy thật.**

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeKingston

def f(x):                                      # hàm mục tiêu của Ví dụ 10
    return (-9 + 13*x[0] + 14*x[1] + 9*x[2]
            - 18*x[0]*x[1] - 18*x[0]*x[2] - 18*x[1]*x[2] + 36*x[0]*x[1]*x[2])

def build_ansatz(theta):                       # Hình 9 của bài báo
    qaoa_depth = len(theta) // 2
    gamma, beta = theta[:qaoa_depth], theta[qaoa_depth:]
    circuit = QuantumCircuit(3)
    for i in range(3):
        circuit.h(i)
    for i in range(qaoa_depth):
        circuit.rz(-4 * gamma[i], 0)
        circuit.rz(-5 * gamma[i], 1)
        circuit.cx(0, 1)
        circuit.cx(1, 2)
        circuit.rz(-9 * gamma[i], 2)
        circuit.cx(1, 2)
        circuit.cx(0, 1)
        for q in range(3):
            circuit.rx(2 * beta[i], q)
    circuit.measure_all()
    return circuit

def mean_f(counts):
    total = sum(counts.values())
    return sum(f([int(v) for v in k[::-1]]) * n for k, n in counts.items()) / total

## 1. Tối ưu tham số trên trình mô phỏng lý tưởng (miễn phí)

Dùng hàm mục tiêu đã sửa (`value +=`) như ở [Notebook 5](../A-examples/05-qaoa-integer-function.ipynb), cùng hạt giống nên tham số tìm được giống nhau.

In [2]:
ideal = AerSimulator()
SEED = 7

def objective(theta, shots=1000):
    counts = ideal.run(build_ansatz(theta), shots=shots, seed_simulator=SEED).result().get_counts()
    return mean_f(counts)

res = minimize(objective, [0.1, 0.1, 0.1, 0.1], method="COBYLA")
theta = res.x
print("tham số tối ưu (gamma1, gamma2, beta1, beta2):", np.round(theta, 3))
print("giá trị trung bình của f khi sampling:", round(res.fun, 3))

tham số tối ưu (gamma1, gamma2, beta1, beta2): [1.176 0.09  0.16  1.151]
giá trị trung bình của f khi sampling: -4.514


## 2. Chọn backend

- Có tài khoản: `QiskitRuntimeService()` đọc thông tin đã lưu bằng `save_account` (hoặc các biến môi trường `QISKIT_IBM_TOKEN`, `QISKIT_IBM_INSTANCE`), rồi `least_busy` chọn máy ít hàng đợi nhất.
- Không có tài khoản: dùng `FakeKingston()`.

In [3]:
USE_REAL = os.environ.get("USE_REAL_IBM", "1") == "1"
try:
    if not USE_REAL:
        raise RuntimeError("USE_REAL_IBM=0")
    service = QiskitRuntimeService()
    backend = service.least_busy(min_num_qubits=3, operational=True, simulator=False)
    ON_REAL = True
    print("Đang dùng máy thật:", backend.name, "|", backend.num_qubits, "qubit")
except Exception as e:
    backend = FakeKingston()
    ON_REAL = False
    print(f"Không kết nối được IBM Quantum ({type(e).__name__}).")
    print("-> Dùng FakeKingston (mô phỏng noise cục bộ):", backend.name, "|", backend.num_qubits, "qubit")

Không kết nối được IBM Quantum (AccountNotFoundError).
-> Dùng FakeKingston (mô phỏng noise cục bộ): fake_kingston | 156 qubit


## 3. Transpile

Máy thật chỉ hiểu một tập native gate (với chip Heron: `rz`, `sx`, `x`, `cz`) và chỉ cho phép CNOT/CZ giữa các qubit nối vật lý với nhau. Bước transpile biến mạch của ta thành mạch "ISA" tương thích. Mạch ba qubit nhỏ của ta nở ra khá nhiều.

In [4]:
logical = build_ansatz(theta)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=1)
isa = pm.run(logical)
print("mạch gốc  : depth", logical.depth(), "| số gate:", dict(logical.count_ops()))
print("mạch ISA  : depth", isa.depth(), "| số gate:", dict(isa.count_ops()))

mạch gốc  : depth 16 | số gate: {'cx': 8, 'rz': 6, 'rx': 6, 'h': 3, 'measure': 3, 'barrier': 1}
mạch ISA  : depth 46 | số gate: {'rz': 33, 'sx': 19, 'cz': 8, 'measure': 3, 'barrier': 1}


## 4. Chạy và so sánh với trình mô phỏng lý tưởng

Ta lấy 1000 mẫu từ backend rồi so với 1000 mẫu từ mô phỏng lý tưởng (cùng tham số). Trên máy thật hoặc mô hình noise, các solution xấu như `111` ($f = 9$) xuất hiện nhiều hơn.

In [5]:
SHOTS = 1000
sampler = Sampler(mode=backend)                # job mode: mỗi lần chạy là một job độc lập
job = sampler.run([isa], shots=SHOTS)
if ON_REAL:
    print("job id:", job.job_id(), "| state:", job.status())
noisy = job.result()[0].data.meas.get_counts()      # thanh ghi tên 'meas' do measure_all() tạo ra

clean = ideal.run(logical, shots=SHOTS, seed_simulator=11).result().get_counts()

print("Khóa | (x1,x2,x3) |  f  | lý tưởng | backend")
for key in sorted(clean, key=lambda k: -clean[k]):
    x = tuple(int(v) for v in key[::-1])
    print(f" {key} | {x} | {f(x):3d} | {clean[key]:8d} | {noisy.get(key, 0):7d}")
print()
print("giá trị trung bình của f -> lý tưởng:", round(mean_f(clean), 2), "| backend:", round(mean_f(noisy), 2))
print("measurement probability được solution tối ưu 000 -> lý tưởng:", clean.get("000", 0) / SHOTS, "| backend:", noisy.get("000", 0) / SHOTS)

Khóa | (x1,x2,x3) |  f  | lý tưởng | backend
 000 | (0, 0, 0) |  -9 |      312 |     261
 110 | (0, 1, 1) |  -4 |      285 |     268
 101 | (1, 0, 1) |  -5 |      166 |     117
 011 | (1, 1, 0) |   0 |      147 |     126
 010 | (0, 1, 0) |   5 |       41 |      97
 100 | (0, 0, 1) |   0 |       27 |      61
 001 | (1, 0, 0) |   4 |       16 |      33
 111 | (1, 1, 1) |   9 |        6 |      37

giá trị trung bình của f -> lý tưởng: -4.46 | backend: -3.06
measurement probability được solution tối ưu 000 -> lý tưởng: 0.312 | backend: 0.261


In [6]:
if ON_REAL:
    # job.usage() trả về 0 cho đến khi IBM tính xong; nếu thấy 0 hãy gọi lại sau ít phút
    print("Thời gian QPU job này đã dùng (giây):", job.usage())
    print("Mức sử dụng của gói hiện tại:", service.usage())
else:
    print("(Chế độ mô phỏng: không có thông tin sử dụng QPU.)")

(Chế độ mô phỏng: không có thông tin sử dụng QPU.)


Nhận xét: noise làm giảm xác suất của solution tối ưu và nâng xác suất của các solution xấu so với mô phỏng lý tưởng. Đây là điều đã đề cập ở Mục 1 của bài báo (kỷ nguyên NISQ). Với mạch lớn hơn hoặc nhiều lớp $p$ hơn, mạch ISA càng sâu và noise càng nặng, nên số layer $p$ lớn không phải lúc nào cũng tốt hơn trên máy thật.